In [2]:
messages = [
    "Когда будет доставка?",
    "Где моя посылка?",
    "Хочу вернуть товар"
]

for massage in messages:
    text = massage.lower()

    if "достав" in text:
        category = "Доставка"

    elif "вернут" in text:
        category = "Возврат"

    else:
        category = "Неизвестно"

    print(f"{text} -> {category}")

когда будет доставка? -> Доставка
где моя посылка? -> Неизвестно
хочу вернуть товар -> Возврат


In [3]:
import pandas as pd

In [4]:
import sklearn

In [5]:
import matplotlib.pyplot as plt

In [6]:
print(sklearn.__version__)

1.9.1


In [7]:
messages_df = pd.read_csv("messages.csv", encoding="utf-8")
messages_df.head()

,message,category
0,Хочу поменять товар на аналогичный,Возврат
1,Как изменить получателя заказа?,Доставка
2,Можно вернуть подаренный товар?,Возврат
3,Где сейчас находится моя посылка?,Доставка
4,Сколько ждать рассмотрения заявления на возврат?,Возврат


In [8]:
messages_df.shape

(135, 2)

In [9]:
messages_df["category"].value_counts()

category
Возврат     45
Доставка    45
Оплата      45
Name: count, dtype: int64

In [10]:
messages_df.isna().sum()

message     0
category    0
dtype: int64

In [11]:
X = messages_df["message"]
y = messages_df["category"]

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer


demo_texts = [
    "Доставка заказа задерживается",
    "Доставка заказа вечером",
    "Возврат заказа оформлен",
]

demo_vectorizer = TfidfVectorizer()
print(type(demo_vectorizer))

demo_matrix = demo_vectorizer.fit_transform(demo_texts)

demo_words = demo_vectorizer.get_feature_names_out()

demo_table = pd.DataFrame(
    demo_matrix.toarray(),
    columns=demo_words,
    index=demo_texts
).round(2)

demo_table


<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


,вечером,возврат,доставка,задерживается,заказа,оформлен
Доставка заказа задерживается,0.00,0.00,0.55,0.72,0.43,0.00
Доставка заказа вечером,0.72,0.00,0.55,0.00,0.43,0.00
Возврат заказа оформлен,0.00,0.65,0.00,0.00,0.39,0.65


In [18]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f" Сообщений для обучения {len(X_train)}")
print(f" Сообщений для тестирования {len(X_test)}")
print(f" Категорий в обучающей выборке {y_train.value_counts()}")
print(f" Категорий в тестовой выборке {y_test.value_counts()}")

 Сообщений для обучения 108
 Сообщений для тестирования 27
 Категорий в обучающей выборке category
Оплата      36
Возврат     36
Доставка    36
Name: count, dtype: int64
 Категорий в тестовой выборке category
Доставка    9
Возврат     9
Оплата      9
Name: count, dtype: int64


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [24]:
model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

print(f"Этапы Pipline {len(model.named_steps)} - {model.named_steps}")


Этапы Pipline 2 - {'tfidf': TfidfVectorizer(), 'classifier': LogisticRegression(max_iter=1000)}


In [25]:
model.fit(X_train, y_train)
print(f'Модель обучена на сообщениях {len(X_train)}')

Модель обучена на сообщениях 108


In [26]:
new_messages = [
    "Курьер привезет заказ сегодня?",
    "Не могу оплатить заказ банковской картой",
    "Можно обменять ботинки на другой размер?",
    "Хочу вернуть товар и получить деньги",
]

In [27]:
new_predictions = model.predict(new_messages)

In [28]:
new_results = pd.DataFrame({
    "Сообщения": new_messages,
    "Предсказанная категория": new_predictions,
})
new_results

,Сообщения,Предсказанная категория
0,Курьер привезет заказ сегодня?,Доставка
1,Не могу оплатить заказ банковской картой,Оплата
2,Можно обменять ботинки на другой размер?,Возврат
3,Хочу вернуть товар и получить деньги,Возврат


In [29]:

difficult_messages = [
    "Не пришел заказ, хочу вернуть деньги",
    "Курьер привез разбитую кружку, хочу получить замену",
    "Подскажите график работы",
]


In [30]:
difficult_predictions = model.predict(difficult_messages)

In [31]:
difficult_results = pd.DataFrame({
    "Сообщения": difficult_messages,
    "Предсказанная категория": difficult_predictions,
})
difficult_results

,Сообщения,Предсказанная категория
0,"Не пришел заказ, хочу вернуть деньги",Возврат
1,"Курьер привез разбитую кружку, хочу получить з...",Возврат
2,Подскажите график работы,Доставка
